- **Goal:** Map JobTech occupation labels to SSYK codes
- **Challenges:** Different naming conventions, granularity mismatch
- **Decision:** Map at `occupation_group_label` level

## Inspect SSYK table

In [0]:
DESCRIBE silver_work_scb.ssyk_mapping;

In [0]:
SELECT *
FROM silver_work_scb.ssyk_mapping
LIMIT 5;

In [0]:
SELECT DISTINCT
    occupation_code,
    occupation_name
FROM silver_work_scb.ssyk_mapping
LIMIT 50;

## Inspect JobTech occupation labels

In [0]:
SELECT DISTINCT
    occupation_group_label,
    occupation_field_label,
    occupation_label
FROM silver_jobtech.job_postings_silver_clean
WHERE occupation_label IS NOT NULL
  AND TRIM(occupation_label) <> ''
LIMIT 50;

In [0]:
SELECT
    occupation_group_label,
    occupation_field_label,
    occupation_label,
    COUNT(*) AS posting_count
FROM silver_jobtech.job_postings_silver_clean
WHERE occupation_label IS NOT NULL
  AND TRIM(occupation_label) <> ''
GROUP BY
    occupation_group_label,
    occupation_field_label,
    occupation_label
ORDER BY posting_count DESC
LIMIT 50;

## Comparison
Based on the insights above we can not safely map `occupation_label` to `SSYK` directly because 
- JobTech data is messy, detailed, and sometimes multi-role: "Systemutvecklare/Programmerare", "Fordonsmekaniker/Bilmekaniker/Personbilsmekaniker"
- SSYK table is: structured, standardized, and hierarchical (codes like 1111, 0210, etc.)

This is a **many-to-many** + **fuzzy mapping** problem.

On the other hand `occupation_group_label` is MUCH closer to `SSYK` than `occupation_label` since it's either already standardized grouped or grouped: `Grundutbildade sjuksköterskor`, `Grundutbildade sjuksköterskor`, `Mjukvaru- och systemutvecklare m.fl.`

## Identify mapping strategy
So we are going to map at GROUP LEVEL using `occupation_group_label → SSYK`

This strategy is more stable since the group labels are consistent, the granularity matches since the SSYK and the group labels are already grouped.

But the SSYK is high level with limited subset
```
0110 Officerare
0210 Specialistofficerare
0310 Soldater m.fl.
1111 Politiker
```
So before mapping we want to check if `silver_work_scb.ssyk_mapping` complete because if it's too small we cannot use it directly.

In [0]:
SELECT DISTINCT occupation_group_label
FROM silver_jobtech.job_postings_silver_clean
WHERE occupation_group_label IS NOT NULL;

In [0]:
SELECT COUNT(*)
FROM silver_work_scb.ssyk_mapping;

In [0]:
SELECT *
FROM silver_work_scb.ssyk_mapping
LIMIT 50;

`432` rows strongly suggests this is a reasonably complete occupation reference table, not just a tiny sample. And from the values, many `occupation_group_label` values from `JobTech` appear to line up very closely with `occupation_name` in `silver_work_scb.ssyk_mapping`.

Howrver one important complication is that some SSYK names contain:

`nivå 1`, `nivå 2`, punctuation differences, and slight naming differences

So a simple exact join will work for some rows, but not all.

We likely need a two-step approach:

- Step 1: Try exact matching on cleaned text.
- Step 2: Handle remaining unmatched occupations manually or with rule-based logic.

In [0]:
SELECT
    COUNT(DISTINCT j.occupation_group_label) AS total_jobtech_groups,
    COUNT(DISTINCT s.occupation_name) AS matched_groups
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name);

In [0]:
SELECT
    j.occupation_group_label,
    s.occupation_code,
    s.occupation_name
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name)
WHERE s.occupation_code IS NOT NULL
ORDER BY j.occupation_group_label;

In [0]:
SELECT
    j.occupation_group_label
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name)
WHERE s.occupation_code IS NULL
ORDER BY j.occupation_group_label;

## Occupation mapping strategy

To harmonize JobTech occupation data with SCB labour-market datasets, the mapping will be performed at the `occupation_group_label` level rather than the `occupation_label` level.

### Why map at group level?
- `occupation_group_label` is more standardized and much closer to SSYK occupation names
- `occupation_label` is often too detailed, inconsistent, or multi-role
- group-level mapping is more stable and analytically defensible

### Initial mapping results
- Distinct JobTech occupation groups: **426**
- Exact SSYK matches found: **363**
- Exact match coverage: **~85%**

This confirms that `occupation_group_label` is a strong basis for harmonization.

### Remaining unmatched cases
The remaining unmatched occupation groups appear to be caused mainly by:
- spelling and abbreviation differences
- punctuation differences
- shortened forms such as `o` instead of `och`
- SSYK categories split into `nivå 1` and `nivå 2`

### Planned mapping approach
1. Exact match between `occupation_group_label` and `occupation_name`
2. Normalized text matching for minor naming differences
3. Manual override rules for ambiguous or level-split occupations

This approach provides a transparent and maintainable bridge between JobTech and SSYK, ready to support the Gold analytics layer.

## Build initial occupation mapping table

This step creates the first version of the occupation mapping bridge between JobTech and SSYK.

### Purpose
The table maps `occupation_group_label` from JobTech to `occupation_code` and `occupation_name` from the SSYK reference table.

### Method
This first version uses **exact matching only** between:
- `occupation_group_label`
- `occupation_name`

### Expected result
- matched rows receive a valid `occupation_code`
- unmatched rows remain with `NULL` in the SSYK columns
- `mapping_method` is set to `exact_match`

This provides the baseline mapping coverage before applying any text normalization or manual override rules.

In [0]:
CREATE OR REPLACE TABLE silver_jobtech.occupation_group_to_ssyk_mapping AS
SELECT DISTINCT
    j.occupation_group_label,
    s.occupation_code,
    s.occupation_name,
    'exact_match' AS mapping_method
FROM (
    SELECT DISTINCT occupation_group_label
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
) j
LEFT JOIN silver_work_scb.ssyk_mapping s
    ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name);

In [0]:
SELECT
    COUNT(*) AS total_groups,
    SUM(CASE WHEN occupation_code IS NOT NULL THEN 1 ELSE 0 END) AS matched_groups,
    SUM(CASE WHEN occupation_code IS NULL THEN 1 ELSE 0 END) AS unmatched_groups
FROM silver_jobtech.occupation_group_to_ssyk_mapping;

## Validate initial mapping coverage

This step validates the first version of the occupation mapping table.

### What is being checked
- total number of distinct JobTech occupation groups `426`
- how many groups matched to SSYK `363`
- how many groups are still unmatched `63`

### Why this matters
This confirms that the initial exact-match mapping table was created correctly and provides a baseline before the next refinement step.

## Normalized matching analysis

This step evaluates how many additional occupation mappings can be recovered using text normalization.

### Why normalization is needed
The unmatched occupation groups are primarily caused by:
- abbreviation differences (`o` vs `och`)
- formatting differences (`m fl` vs `m.fl.`)
- punctuation inconsistencies

### Approach
Both JobTech and SSYK occupation names are normalized by:
- converting to lowercase
- replacing common abbreviations
- removing punctuation

The normalized values are then compared to identify additional matches.

### Important
This step does not modify the mapping table.  
It is used only to evaluate potential improvements before updating the mapping.

In [0]:
WITH jobtech_groups AS (
    SELECT DISTINCT
        occupation_group_label,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_group_label), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_group
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
),
ssyk_groups AS (
    SELECT DISTINCT
        occupation_code,
        occupation_name,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_name), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_name
    FROM silver_work_scb.ssyk_mapping
)
SELECT
    j.occupation_group_label,
    s.occupation_code,
    s.occupation_name
FROM jobtech_groups j
LEFT JOIN ssyk_groups s
    ON j.normalized_group = s.normalized_name
WHERE s.occupation_code IS NOT NULL
ORDER BY j.occupation_group_label;

## Quantify recoverable matches from normalization

This step measures how many previously unmatched JobTech occupation groups can be mapped to SSYK after applying text normalization.

### Purpose
The goal is to isolate only the additional matches recovered through normalization, beyond the initial exact-match baseline.

### Why this matters
This helps distinguish between:
- mappings already handled by exact matching
- mappings newly recovered through normalization
- mappings that still require manual review or override rules

In [0]:
WITH unmatched_jobtech AS (
    SELECT
        occupation_group_label,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_group_label), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_group
    FROM silver_jobtech.occupation_group_to_ssyk_mapping
    WHERE occupation_code IS NULL
),
normalized_ssyk AS (
    SELECT DISTINCT
        occupation_code,
        occupation_name,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_name), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_name
    FROM silver_work_scb.ssyk_mapping
)
SELECT
    COUNT(*) AS previously_unmatched_groups,
    SUM(CASE WHEN s.occupation_code IS NOT NULL THEN 1 ELSE 0 END) AS recovered_by_normalization,
    SUM(CASE WHEN s.occupation_code IS NULL THEN 1 ELSE 0 END) AS still_unmatched
FROM unmatched_jobtech u
LEFT JOIN normalized_ssyk s
    ON u.normalized_group = s.normalized_name;

## Review normalized recoveries

This step lists the occupation groups that were not matched by exact comparison but were successfully matched after text normalization.

### Purpose
Reviewing these rows confirms which normalization rules are working and identifies the specific naming differences between JobTech and SSYK.

### Expected patterns
The recovered matches are typically caused by:
- `o` instead of `och`
- `m fl` instead of `m.fl.`
- punctuation differences
- shortened wording variants

These rows are good candidates to update into the mapping table with `mapping_method = 'normalized_match'`.

In [0]:
WITH unmatched_jobtech AS (
    SELECT
        occupation_group_label,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_group_label), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_group
    FROM silver_jobtech.occupation_group_to_ssyk_mapping
    WHERE occupation_code IS NULL
),
normalized_ssyk AS (
    SELECT DISTINCT
        occupation_code,
        occupation_name,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_name), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_name
    FROM silver_work_scb.ssyk_mapping
)
SELECT
    u.occupation_group_label,
    s.occupation_code,
    s.occupation_name
FROM unmatched_jobtech u
LEFT JOIN normalized_ssyk s
    ON u.normalized_group = s.normalized_name
WHERE s.occupation_code IS NOT NULL
ORDER BY u.occupation_group_label;

## Apply normalized occupation mapping

This step updates the occupation mapping table by incorporating matches identified through text normalization.

### What is being done
- Existing exact matches are preserved
- Previously unmatched occupation groups are re-evaluated using normalized text
- Matching rows are updated with SSYK codes
- `mapping_method` is set to:
  - `exact_match` for original matches
  - `normalized_match` for recovered matches

### Why this matters
This improves mapping coverage while maintaining traceability of how each mapping was created.

### Result
The mapping table now contains both exact and normalized matches, increasing overall coverage and readiness for Gold layer modeling.

In [0]:
CREATE OR REPLACE TABLE silver_jobtech.occupation_group_to_ssyk_mapping AS
WITH jobtech_groups AS (
    SELECT DISTINCT
        occupation_group_label,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_group_label), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_group
    FROM silver_jobtech.job_postings_silver_clean
    WHERE occupation_group_label IS NOT NULL
),
ssyk_groups AS (
    SELECT DISTINCT
        occupation_code,
        occupation_name,
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(TRIM(occupation_name), '\\bm fl\\b', 'm.fl.'),
                    '\\bo\\b', 'och'
                ),
                '[\\.,"]',
                ''
            )
        ) AS normalized_name
    FROM silver_work_scb.ssyk_mapping
),
exact_matches AS (
    SELECT DISTINCT
        j.occupation_group_label,
        s.occupation_code,
        s.occupation_name,
        'exact_match' AS mapping_method
    FROM (
        SELECT DISTINCT occupation_group_label
        FROM silver_jobtech.job_postings_silver_clean
        WHERE occupation_group_label IS NOT NULL
    ) j
    LEFT JOIN silver_work_scb.ssyk_mapping s
        ON TRIM(j.occupation_group_label) = TRIM(s.occupation_name)
),
normalized_matches AS (
    SELECT DISTINCT
        j.occupation_group_label,
        s.occupation_code,
        s.occupation_name,
        'normalized_match' AS mapping_method
    FROM jobtech_groups j
    LEFT JOIN ssyk_groups s
        ON j.normalized_group = s.normalized_name
)
SELECT
    COALESCE(e.occupation_group_label, n.occupation_group_label) AS occupation_group_label,
    COALESCE(e.occupation_code, n.occupation_code) AS occupation_code,
    COALESCE(e.occupation_name, n.occupation_name) AS occupation_name,
    CASE
        WHEN e.occupation_code IS NOT NULL THEN 'exact_match'
        WHEN n.occupation_code IS NOT NULL THEN 'normalized_match'
        ELSE 'unmatched'
    END AS mapping_method
FROM exact_matches e
FULL OUTER JOIN normalized_matches n
    ON e.occupation_group_label = n.occupation_group_label;

In [0]:
SELECT
    mapping_method,
    COUNT(*) AS count
FROM silver_jobtech.occupation_group_to_ssyk_mapping
GROUP BY mapping_method;

## Validate updated mapping coverage

This step confirms the final mapping coverage after applying normalized matches.

### What is being checked
- total occupation groups
- matched occupation groups
- unmatched occupation groups
- coverage percentage

### Why this matters
This provides the final quality checkpoint for the occupation harmonization step before deciding whether manual overrides are needed.

In [0]:
SELECT
    COUNT(*) AS total_groups,
    SUM(CASE WHEN occupation_code IS NOT NULL THEN 1 ELSE 0 END) AS matched_groups,
    SUM(CASE WHEN occupation_code IS NULL THEN 1 ELSE 0 END) AS unmatched_groups,
    ROUND(
        100.0 * SUM(CASE WHEN occupation_code IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS match_coverage_pct
FROM silver_jobtech.occupation_group_to_ssyk_mapping;

## Milestone

Enhanced occupation mapping using normalization techniques.

### Results
- Total occupation groups: 426
- Exact matches: 363
- Normalized matches: 10
- Unmatched: 53
- Coverage: 87.56%

### Key improvements
- Resolved abbreviation mismatches (e.g. "o" → "och")
- Standardized formatting differences (e.g. "m fl" → "m.fl.")
- Preserved traceability through `mapping_method`

The mapping is now sufficiently robust for Gold layer modeling. Remaining unmatched cases can be handled later via manual overrides if needed.